# Network Topology Test

Create a `Qwen3DecoderLayer` and convert it to an `NxGraph` with `NxTracer`.

In [3]:
import json
from pathlib import Path

import torch
from nandmachine.config.hbm_hbf_architecture import validate_hbm_hbf_architecture_or_raise
from nandmachine.config.model_config import Qwen3ModelConfig

from nandmachine.frontend.core.graph.base import NxGraph, NxTracer
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.qwen3 import Qwen3DecoderLayer

config_path = Path("model_cards/qwen3-8B.json")
DEVICE_NAME = "H200_SXM"
COMPILE_MODE = "heuristic-GPU"
config = Qwen3ModelConfig.from_dict(json.loads(config_path.read_text()))

simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
    "hbf_sram_intermediate_buffer": True,
    "memory_architecture": {
        "mode": "hbm_only",
        # "mode": "csi",
        # "mode": "cli", "hbm_stacks": 2, "hbf_stacks": 3,
    },
}
validated_memory_architecture = validate_hbm_hbf_architecture_or_raise(
    simulator_config["device_name"],
    simulator_config["memory_architecture"],
)

print("Config loaded successfully")
print(f"hidden_size={config.hidden_size}")
print(f"num_attention_heads={config.num_attention_heads}")
print(f"intermediate_size={config.intermediate_size}")
print("simulator config:", simulator_config)
print("validated memory architecture:", validated_memory_architecture)


Config loaded successfully
hidden_size=4096
num_attention_heads=32
intermediate_size=12288


In [4]:
with torch.device("meta"):
    layer = Qwen3DecoderLayer(config)

layer.eval()
tracer = NxTracer()
nx_graph = tracer.trace(layer)
graph_module = torch.fx.GraphModule(layer, nx_graph)

print(type(nx_graph))
print(f"num_nodes={len(tuple(nx_graph.nodes))}")
print(graph_module)

<class 'nandmachine.frontend.core.graph.base.NxGraph'>
num_nodes=28
GraphModule(
  (input_layernorm): RMSNorm()
  (self_attn): Module(
    (qkv_proj): QKVParallelLinear()
    (q_norm): RMSNorm()
    (k_norm): RMSNorm()
    (rotary_emb): RotaryEmbedding()
    (attn): Attention()
    (o_proj): RowParallelLinear()
  )
  (post_attention_layernorm): RMSNorm()
  (mlp): Module(
    (gate_up_proj): MergedColumnParallelLinear()
    (act_fn): SiluAndMul()
    (down_proj): RowParallelLinear()
  )
)



def forward(self, hidden_states : _torch_Tensor_):
    input_layernorm = self.input_layernorm(hidden_states)
    self_attn_qkv_proj = self.self_attn.qkv_proj(input_layernorm);  input_layernorm = None
    split = self_attn_qkv_proj.split((4096, 1024, 1024), dim = -1);  self_attn_qkv_proj = None
    getitem = split[0]
    getitem_1 = split[1]
    getitem_2 = split[2];  split = None
    unflatten = getitem.unflatten(-1, (32, 128));  getitem = None
    unflatten_1 = getitem_1.unflatten(-1, (8, 128));  g

In [5]:
# assert isinstance(nx_graph, NxGraph)

# call_module_targets = [
#     node.target
#     for node in nx_graph.nodes
#     if node.op == "call_module"
# ]

# print("call_module targets:")
# for target in call_module_targets:
#     print(target)

# print("\nGenerated Graph:")
# print(nx_graph)

In [6]:
normalize_tracer = NxTracer()
normalized_graph = normalize_tracer.trace(layer)
normalized_graph_module = torch.fx.GraphModule(layer, normalized_graph)
normalize_pass = NormalizePass()

normalize_pass.transform(normalized_graph_module)

print(f"normalized num_nodes={len(tuple(normalized_graph_module.graph.nodes))}")
print("Normalized nodes:")
for node in normalized_graph_module.graph.nodes:
    print(f"{node.name:24} {node.op:14} {node.target}")

print("\nNormalized hook modules:")
for node in normalized_graph_module.graph.nodes:
    hook_module = node.meta.get("hook_module")
    
    # print(node.meta)
    if hook_module is not None:
        print(f"{node.name:24} {hook_module}")

# print("\nNormalized Graph:")
# print(normalized_graph_module.graph)


normalized num_nodes=15
Normalized nodes:
hidden_states            placeholder    hidden_states
input_layernorm          call_module    input_layernorm
self_attn_qkv_proj       call_module    self_attn.qkv_proj
self_attn_q_norm         call_module    self_attn.q_norm
self_attn_k_norm         call_module    self_attn.k_norm
self_attn__dummy_positions get_attr       self_attn._dummy_positions
self_attn_rotary_emb     call_module    self_attn.rotary_emb
self_attn_attn           call_module    self_attn.attn
self_attn_o_proj         call_module    self_attn.o_proj
add                      call_function  <built-in function add>
post_attention_layernorm call_module    post_attention_layernorm
mlp_gate_up_proj         call_module    mlp.gate_up_proj
mlp_act_fn               call_module    mlp.act_fn
mlp_down_proj            call_module    mlp.down_proj
add_1                    call_function  <built-in function add>

Normalized hook modules:
input_layernorm          RMSNorm()
self_attn_qkv_pro